# 02 — Feature Engineering
**NBA Player Performance & Trade Value Predictor**  
DATA 300 · Spring 2026 · Anh Vu & Drew Norton

---

### Goals of this notebook
1. Load the combined dataset from notebook 01
2. Build **lag features** (previous season stats as predictors)
3. Build **3-year rolling features** (rolling mean + trend)
4. Build **age curve features** (polynomial age, years from peak)
5. Build **salary efficiency features**
6. Construct the **regression design matrix** — one row per player-season to predict
7. Save `data/processed/players_features.csv` for notebook 03

### Key design decision
Both single-season lag features AND 3-year rolling features are built.  
Notebook 03 will test both and keep whichever performs better in cross-validation.


---
## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
})

print('Imports OK')

---
## 1. Load Data

In [ ]:
df = pd.read_csv('data/processed/players_combined.csv')
print(f'Loaded: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Unique players: {df["Player"].nunique():,}')

# Sort — critical for lag/rolling features to work correctly
df = df.sort_values(['Player', 'season']).reset_index(drop=True)

# Columns we will build features from
STAT_COLS = [
    'PER', 'TS_pct', 'BPM', 'VORP', 'WS', 'WS_per48',
    'ast_pct', 'reb_pct', 'blk_pct', 'stl_pct', 'tov_pct', 'usg_pct',
    'PTS', 'TRB', 'AST', 'STL', 'BLK', 'MP', 'G'
]
# Only keep stat cols that actually exist in the data
STAT_COLS = [c for c in STAT_COLS if c in df.columns]

# Target variables for regression
TARGETS = ['PER', 'PTS', 'WS']

print(f'\nStat columns available: {STAT_COLS}')
print(f'Regression targets: {TARGETS}')

---
## 2. Lag Features (Previous Season Stats)

The simplest predictor of next season's performance is **last season's performance**.  
For each stat, we create `{stat}_lag1` = that player's value in the previous season.

**Important:** lag features shift within each player's history.  
If a player has no previous season in our dataset (first season), lag = NaN → those rows get dropped later.

In [ ]:
print('Building lag features...')

for col in STAT_COLS:
    if col in df.columns:
        df[f'{col}_lag1'] = df.groupby('Player')[col].shift(1)

# Also lag age and salary — these matter for the regression
df['Age_lag1']           = df.groupby('Player')['Age'].shift(1)
df['salary_lag1']        = df.groupby('Player')['salary'].shift(1)
df['salary_pct_cap_lag1']= df.groupby('Player')['salary_pct_cap'].shift(1)

lag_cols = [c for c in df.columns if c.endswith('_lag1')]
print(f'Created {len(lag_cols)} lag features')

# Quick check — LeBron James lag1 PER should look like a shifted version
if 'LeBron James' in df['Player'].values:
    lebron = df[df['Player'] == 'LeBron James'][['season', 'PER', 'PER_lag1']]
    print('\nLeBron James — PER vs lag1 check:')
    print(lebron.to_string(index=False))

---
## 3. Rolling Features (3-Year Window)

Rolling features capture **trends** that single-season lags miss:  
- A player improving year-over-year  
- A player declining after an injury  
- Noise in a single outlier season getting smoothed out  

We compute:
- `{stat}_roll3_mean` — 3-season rolling average (up to but not including current season)
- `{stat}_roll3_std` — rolling standard deviation (consistency signal)
- `{stat}_delta1` — year-over-year change (trend direction)

In [ ]:
print('Building rolling features...')

KEY_ROLLING_COLS = ['PER', 'BPM', 'VORP', 'WS', 'PTS', 'TS_pct', 'usg_pct']
KEY_ROLLING_COLS = [c for c in KEY_ROLLING_COLS if c in df.columns]

for col in KEY_ROLLING_COLS:
    # Rolling mean: min_periods=1 so players with only 1-2 prior seasons still get a value
    # shift(1) ensures we never include the current season in the window
    df[f'{col}_roll3_mean'] = (
        df.groupby('Player')[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    )
    
    # Rolling std — players with < 2 prior seasons get NaN here, that's fine
    df[f'{col}_roll3_std'] = (
        df.groupby('Player')[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=2).std())
    )
    
    # Year-over-year delta: current - previous
    df[f'{col}_delta1'] = df.groupby('Player')[col].diff(1)

rolling_cols = [c for c in df.columns if 'roll3' in c or 'delta1' in c]
print(f'Created {len(rolling_cols)} rolling/delta features')

# Verify with a known player
if 'Stephen Curry' in df['Player'].values:
    curry = df[df['Player'] == 'Stephen Curry'][['season', 'PER', 'PER_lag1', 'PER_roll3_mean', 'PER_delta1']]
    print('\nSteph Curry — PER feature check:')
    print(curry.to_string(index=False))

---
## 4. Age Curve Features

NBA players follow a predictable career arc:  
- **Rising phase:** 18–26 (learning, athleticism improving)  
- **Peak:** ~27–29  
- **Declining phase:** 30+ (athleticism fades, efficiency may hold for skill players)

A linear age term can't capture this curve.  
We use `Age` + `Age²` so the model can fit the non-linear shape.

In [ ]:
# Polynomial age features
df['age_sq']         = df['Age'] ** 2
df['years_from_peak']= 27 - df['Age']          # positive = ascending, negative = declining
df['past_prime']     = (df['Age'] > 30).astype(int)  # binary flag

# Seasons in the league (proxy for experience)
df['seasons_in_data'] = df.groupby('Player').cumcount() + 1

print('Age curve features added.')

# Visualize the age curve for PER to validate the non-linearity
age_curve = df.groupby('Age')['PER'].agg(['mean', 'count']).reset_index()
age_curve = age_curve[age_curve['count'] >= 30]  # only ages with enough data

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(age_curve['Age'], age_curve['mean'], 'o-', color='#1F3864', linewidth=2.5, markersize=6)
ax.axvline(27, color='#E8A020', linestyle='--', linewidth=1.5, label='Assumed peak (age 27)')
ax.fill_between(age_curve['Age'], age_curve['mean'], alpha=0.1, color='#1F3864')
ax.set_xlabel('Player Age', fontsize=12)
ax.set_ylabel('Average PER', fontsize=12)
ax.set_title('NBA Age Curve — Average PER by Age (2015–2024)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/age_curve.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/age_curve.png')

---
## 5. Salary Efficiency Features

In [ ]:
# Dollars per unit of production (lower = more efficient)
eps = 1e-6  # avoid division by zero

df['dollars_per_PER']  = df['salary'] / (df['PER']  + eps)
df['dollars_per_VORP'] = df['salary'] / (df['VORP'] + eps + 3)  # shift VORP so no negatives
df['dollars_per_WS']   = df['salary'] / (df['WS']   + eps + 1)

# Salary tier: rookie / mid-level / max contract
# Based on % of cap: <5% = rookie deal, 5-20% = mid, >20% = max
def salary_tier(pct):
    if pd.isna(pct):    return 'unknown'
    if pct < 0.05:      return 'rookie'
    elif pct < 0.20:    return 'mid_level'
    else:               return 'max'

df['salary_tier'] = df['salary_pct_cap'].apply(salary_tier)

print('Salary efficiency features added.')
print('\nSalary tier distribution:')
print(df['salary_tier'].value_counts().to_string())

---
## 6. Build Regression Design Matrix

For regression, each row represents:  
> **"Given everything we know up to season T, predict season T+1"**

So the **target** is the current season's stat, and the **features** are from the previous season(s).  
We construct this by shifting targets forward one season per player.

In [ ]:
# Create next-season target columns
for target in TARGETS:
    if target in df.columns:
        df[f'{target}_next'] = df.groupby('Player')[target].shift(-1)

TARGET_COLS = [f'{t}_next' for t in TARGETS if f'{t}_next' in df.columns]
print(f'Target columns created: {TARGET_COLS}')

# Define feature sets
# --- Single-season feature set (uses only lag1) ---
FEATURES_LAG1 = (
    [f'{c}_lag1' for c in STAT_COLS if f'{c}_lag1' in df.columns] +
    ['Age', 'age_sq', 'years_from_peak', 'past_prime', 'seasons_in_data',
     'salary_pct_cap_lag1']
)

# --- Rolling feature set (uses lag1 + rolling means + deltas) ---
FEATURES_ROLLING = (
    [f'{c}_lag1' for c in STAT_COLS if f'{c}_lag1' in df.columns] +
    [f'{c}_roll3_mean' for c in KEY_ROLLING_COLS if f'{c}_roll3_mean' in df.columns] +
    [f'{c}_delta1' for c in KEY_ROLLING_COLS if f'{c}_delta1' in df.columns] +
    ['Age', 'age_sq', 'years_from_peak', 'past_prime', 'seasons_in_data',
     'salary_pct_cap_lag1']
)

# Remove duplicates while preserving order
FEATURES_LAG1    = list(dict.fromkeys(FEATURES_LAG1))
FEATURES_ROLLING = list(dict.fromkeys(FEATURES_ROLLING))

# Only keep features that exist in the dataframe
FEATURES_LAG1    = [f for f in FEATURES_LAG1    if f in df.columns]
FEATURES_ROLLING = [f for f in FEATURES_ROLLING if f in df.columns]

print(f'\nLag1 feature set size:    {len(FEATURES_LAG1)} features')
print(f'Rolling feature set size: {len(FEATURES_ROLLING)} features')

### 6a. Build the Regression Dataset

Drop rows where:
- Any target is missing (player's last season — no "next season" to predict)
- All lag features are missing (player's first season — no history to draw from)

In [ ]:
# We need at least the lag1 features + all targets to be non-null
required_cols = TARGET_COLS + ['PER_lag1']  # PER_lag1 as proxy for "has prior season"
df_reg = df.dropna(subset=required_cols).copy()

print(f'Regression dataset: {len(df_reg):,} rows (dropped {len(df) - len(df_reg):,} first/last seasons)')
print(f'Seasons in regression data: {sorted(df_reg["season"].unique())}')

# Fill remaining NaN in features with column median (for rolling std which needs 2+ seasons)
for col in FEATURES_ROLLING:
    if df_reg[col].isna().sum() > 0:
        median_val = df_reg[col].median()
        df_reg[col] = df_reg[col].fillna(median_val)

print(f'\nNaN remaining in features after imputation: {df_reg[FEATURES_ROLLING].isna().sum().sum()}')

---
## 7. Feature Correlation Analysis

Check which features are most correlated with our targets.  
High correlation = strong predictive signal.  
Also flags multicollinearity issues (features correlated with each other).

In [ ]:
# Top correlations with each target
print('Top 10 features correlated with each target:\n')

for target in TARGET_COLS:
    corrs = df_reg[FEATURES_ROLLING + [target]].corr()[target].drop(target)
    top = corrs.abs().nlargest(10)
    print(f'--- {target} ---')
    for feat, val in top.items():
        print(f'  {feat:<35} r = {corrs[feat]:+.3f}')
    print()

In [ ]:
# Correlation heatmap of key features vs targets
VIZ_FEATURES = [
    'PER_lag1', 'BPM_lag1', 'VORP_lag1', 'WS_lag1',
    'PER_roll3_mean', 'BPM_roll3_mean', 'PER_delta1',
    'Age', 'years_from_peak'
]
VIZ_FEATURES = [f for f in VIZ_FEATURES if f in df_reg.columns]

corr_matrix = df_reg[VIZ_FEATURES + TARGET_COLS].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.zeros_like(corr_matrix, dtype=bool)
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}, vmin=-1, vmax=1
)
ax.set_title('Feature–Target Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → figures/correlation_heatmap.png')

---
## 8. Train/Test Split Strategy

**Important:** we do NOT use random train/test split here.  
NBA data is a time series — randomly splitting would leak future data into training.

Strategy: **train on 2016–2022, test on 2023–2024**  
This simulates real-world usage: train on history, predict the most recent seasons.

In [ ]:
TRAIN_SEASONS = list(range(2016, 2023))  # 2015-16 through 2021-22
TEST_SEASONS  = [2023, 2024]             # 2022-23 and 2023-24

df_train = df_reg[df_reg['season'].isin(TRAIN_SEASONS)].copy()
df_test  = df_reg[df_reg['season'].isin(TEST_SEASONS)].copy()

print(f'Train set: {len(df_train):,} rows | seasons {TRAIN_SEASONS[0]}–{TRAIN_SEASONS[-1]}')
print(f'Test set:  {len(df_test):,}  rows | seasons {TEST_SEASONS}')
print(f'Train/test split: {len(df_train)/(len(df_train)+len(df_test))*100:.0f}% / {len(df_test)/(len(df_train)+len(df_test))*100:.0f}%')

---
## 9. Save Feature Dataset

In [ ]:
# Save full regression-ready dataset
df_reg.to_csv('data/processed/players_features.csv', index=False)

# Save feature lists so notebook 03 can import them directly
import json
feature_config = {
    'FEATURES_LAG1':    FEATURES_LAG1,
    'FEATURES_ROLLING': FEATURES_ROLLING,
    'TARGET_COLS':      TARGET_COLS,
    'TARGETS':          TARGETS,
    'TRAIN_SEASONS':    TRAIN_SEASONS,
    'TEST_SEASONS':     TEST_SEASONS,
}
with open('data/processed/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print(f'Saved → data/processed/players_features.csv  ({len(df_reg):,} rows, {df_reg.shape[1]} cols)')
print(f'Saved → data/processed/feature_config.json')
print(f'\nLag1 features:    {len(FEATURES_LAG1)}')
print(f'Rolling features: {len(FEATURES_ROLLING)}')
print(f'Targets:          {TARGET_COLS}')
print('\n--- READY FOR NOTEBOOK 03 ---')

---
## Summary

| Feature Group | Count | Key Features |
|---|---|---|
| Lag1 (previous season) | ~20 | PER_lag1, BPM_lag1, VORP_lag1, PTS_lag1 |
| Rolling 3yr mean | ~7 | PER_roll3_mean, BPM_roll3_mean, WS_roll3_mean |
| Rolling 3yr std | ~7 | PER_roll3_std (consistency) |
| Year-over-year delta | ~7 | PER_delta1 (trend direction) |
| Age curve | 5 | Age, age_sq, years_from_peak, past_prime |
| Salary | 2 | salary_pct_cap_lag1, salary_tier |

**Train/Test split:** temporal — train 2016–2022, test 2023–2024  
**Targets:** PER_next, PTS_next, WS_next (three separate regression problems)